# A/B Testing: Bike Sharing Model

This notebook performs A/B testing by sending live prediction requests to a single Databricks Model Serving endpoint configured with traffic splitting across Champion and Challenger models.

Steps:
- Send random prediction requests
- Capture which model served each prediction
- Store results in a Delta table for later analysis


In [0]:
import requests
import json
import pandas as pd
import random
import time

In [0]:
# Step 1: Authenticate to Databricks by retrieving the notebook's API token
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Step 2: Define the workspace URL and model serving endpoint URL
WORKSPACE_URL = "https://adb-xxxxxxxxxxxxxxx.xx.azuredatabricks.net"  # your workspace
ENDPOINT = f"{WORKSPACE_URL}/serving-endpoints/abooth_bike_serving_endpoint/invocations"

# Step 3: Set the HTTP headers, including the authorization token
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

## Define function to generate random inputs

In [0]:
# Helper function to generate random input data
def generate_random_payload():
    new_data = pd.DataFrame({
        "season": [random.choice([1, 2, 3, 4])],
        "yr": [random.choice([0, 1])],
        "mnth": [random.randint(1, 12)],
        "hr": [random.randint(0, 23)],
        "holiday": [random.choice([0, 1])],
        "weekday": [random.randint(0, 6)],
        "workingday": [random.choice([0, 1])],
        "weathersit": [random.randint(1, 4)],
        "temp": [random.uniform(0.2, 0.8)],
        "atemp": [random.uniform(0.2, 0.8)],
        "hum": [random.uniform(0.3, 0.9)],
        "windspeed": [random.uniform(0.05, 0.4)]
    })
    return {"dataframe_split": new_data.to_dict(orient="split")}, new_data

## Run A/B Simulation

In [0]:
# Track results
results = []

# Simulate N requests
N = 100  # Number of simulated requests

for _ in range(N):
    payload, input_data = generate_random_payload()

    response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))
    
    if response.status_code == 200:
        prediction = response.json()["predictions"][0]

        served_model = response.headers.get("served-model-name")
        prediction_date = response.headers.get("date")

        results.append({
            "timestamp": time.time(),
            "input_features": json.dumps(input_data.to_dict(orient="records")[0]),
            "prediction": prediction,
            "served_model_name": served_model,
            "prediction_date": prediction_date
        })
    else:
        print(f"Error invoking endpoint: {response.text}")

In [0]:
# Create DataFrame
ab_test_df = pd.DataFrame(results)

# Convert to Spark DataFrame
ab_test_spark_df = spark.createDataFrame(ab_test_df)

# Save to Delta Table
ab_test_spark_df.write.mode("overwrite").saveAsTable("alexander_booth.default.bike_model_ab_test_results")

print("✅ A/B Test results saved to Delta table")
display(ab_test_spark_df.head(5))

✅ A/B Test results saved to Delta table


timestamp,input_features,prediction,served_model_name,prediction_date
1.7453583616875143E9,"{""season"": 1, ""yr"": 0, ""mnth"": 9, ""hr"": 7, ""holiday"": 0, ""weekday"": 6, ""workingday"": 0, ""weathersit"": 4, ""temp"": 0.4054114975791753, ""atemp"": 0.6421854694978798, ""hum"": 0.7427222498451148, ""windspeed"": 0.08524485615338118}",25.92886173759762,champion_model,"Tue, 22 Apr 2025 21:46:01 GMT"
1.745358362056281E9,"{""season"": 4, ""yr"": 0, ""mnth"": 10, ""hr"": 7, ""holiday"": 0, ""weekday"": 3, ""workingday"": 0, ""weathersit"": 2, ""temp"": 0.7468309968307285, ""atemp"": 0.6855693836757797, ""hum"": 0.7754996550916358, ""windspeed"": 0.1998177236837509}",98.16763305664062,challenger_model,"Tue, 22 Apr 2025 21:46:02 GMT"
1.7453583623756418E9,"{""season"": 3, ""yr"": 0, ""mnth"": 7, ""hr"": 14, ""holiday"": 0, ""weekday"": 5, ""workingday"": 0, ""weathersit"": 1, ""temp"": 0.7555386951332961, ""atemp"": 0.7410936333378197, ""hum"": 0.7997244820570224, ""windspeed"": 0.19907689095533548}",286.82448089011064,champion_model,"Tue, 22 Apr 2025 21:46:02 GMT"
1.7453583624320648E9,"{""season"": 4, ""yr"": 1, ""mnth"": 8, ""hr"": 0, ""holiday"": 0, ""weekday"": 0, ""workingday"": 0, ""weathersit"": 3, ""temp"": 0.658866297042231, ""atemp"": 0.7881427120965387, ""hum"": 0.7531961704789121, ""windspeed"": 0.06930735680178947}",62.21027096125678,champion_model,"Tue, 22 Apr 2025 21:46:02 GMT"
1.7453583624923794E9,"{""season"": 2, ""yr"": 1, ""mnth"": 7, ""hr"": 21, ""holiday"": 1, ""weekday"": 2, ""workingday"": 0, ""weathersit"": 3, ""temp"": 0.3202612989255639, ""atemp"": 0.23971347963126505, ""hum"": 0.43417730826805223, ""windspeed"": 0.08591254456518807}",74.28166878816677,champion_model,"Tue, 22 Apr 2025 21:46:02 GMT"


## Analyze Results

In [0]:
%sql
-- Analyze the results of the A/B test
SELECT 
  served_model_name, 
  COUNT(*) AS total_predictions, 
  AVG(prediction) AS avg_prediction
FROM alexander_booth.default.bike_model_ab_test_results
GROUP BY served_model_name
ORDER BY served_model_name;

served_model_name,total_predictions,avg_prediction
challenger_model,19,154.49574121675994
champion_model,81,188.89760044497646


## Update Traffic Routing

In [0]:
# Step 1: Define the full new config
new_config_payload = {
    "served_models": [
        {
            "name": "champion_model",
            "model_name": "abooth_delta_share.default.bike_sharing_uc_model",
            "model_version": "2",
            "workload_type": "CPU",
            "workload_size": "Small",
            "scale_to_zero_enabled": True
        },
        {
            "name": "challenger_model",
            "model_name": "abooth_delta_share.default.bike_sharing_uc_model",
            "model_version": "3",
            "workload_type": "CPU",
            "workload_size": "Small",
            "scale_to_zero_enabled": True
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": "champion_model",
                "traffic_percentage": 20 # Gradually decrease traffic to champion_model
            },
            {
                "served_model_name": "challenger_model",
                "traffic_percentage": 80 # Gradually increase traffic to challenger_model
            }
        ]
    }
}

# Step 2: PUT to update the endpoint config
ENDPOINT_NAME = "abooth_bike_serving_endpoint"
put_url = f"{WORKSPACE_URL}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config"

response = requests.put(put_url, headers=headers, data=json.dumps(new_config_payload))

# Step 3: Check result
if response.status_code == 200:
    print("✅ Successfully updated endpoint traffic and config!")
else:
    print(f"❌ Failed to update config: {response.text}")

✅ Successfully updated endpoint traffic and config!


# Summary

In this notebook, we completed the A/B testing workflow by:

- Sending live prediction requests to a single serving endpoint configured with traffic splitting across Champion and Challenger models.
- Capturing and logging model predictions, input features, timestamps, and model versions into a Delta table for analysis.
- Dynamically adjusting traffic percentages to gradually promote the Challenger model based on performance.
- Demonstrating how Databricks Model Serving, Delta Sharing, and Unity Catalog enable seamless, production-grade model experimentation across workspaces and clouds.

## Recommended Next Steps

- **Analyze Model Performance**: Use the logged Delta table to compare accuracy, latency, or business outcomes between models in greater depth.
- **Promote Winning Model**: Shift 100% of traffic to the best-performing model using traffic configuration updates.
- **Archive Older Versions**: Clean up endpoint configurations by archiving models that are no longer needed.
- **Automate with CI/CD**: Integrate model promotion logic into automated MLOps pipelines based on A/B test results.

By following this workflow, organizations can operationalize model experimentation and iteration with minimal overhead, while maintaining full governance and auditability through Unity Catalog and Delta Sharing.
